# MiniLogic и MiniProlog

Собственные парсеры, таблицы истинности, правила Хорна, унификация и SLD-поиск. Ноутбук создаёт два самостоятельных Python-модуля и запускает учебные сценарии.

## Запуск
Python 3.11+. Выполняйте ячейки сверху вниз в новой сессии. Пути к внешним данным задаются в ячейке конфигурации. Исходные данные не подменяются синтетическими.

## Статус проверки
Обновлено 27 сентября 2026. Старые выводы очищены: они не подтверждают результаты исправленного кода. Подробности локальной проверки — в `VALIDATION.md` в корне репозитория.

## Ограничения
MiniProlog использует поиск в глубину с явным лимитом. При достижении лимита отсутствие доказательства не объявляется ложностью. Сценарий перелёта — условный пример правил, не проверка реальных требований к поездкам. Это учебное подмножество Prolog.


## Модуль MiniLogic


In [ ]:
from pathlib import Path
Path('minilogic.py').write_text('# -*- coding: utf-8 -*-\n\nimport sys\nimport itertools\nimport re\nfrom dataclasses import dataclass\nfrom typing import List, Dict, Tuple, Optional, Set\n\n#лексер булевой логики\nTOKEN_SPEC = [\n    ("SKIP",   r"[ \\t\\r\\n]+"),\n    ("ARROW",  r"->"),\n    ("NOT",    r"!"),\n    ("AND",    r"\\*"),\n    ("OR",     r"\\+"),\n    ("LP",     r"\\("),\n    ("RP",     r"\\)"),\n    ("ID",     r"[A-Za-z_][A-Za-z0-9_]*"),\n    ("MISMATCH", r"."),\n]\nTOKEN_RE = re.compile("|".join(f"(?P<{name}>{pattern})" for name, pattern in TOKEN_SPEC))\n\n@dataclass(frozen=True)\nclass Tok:\n    typ: str\n    val: str\n\ndef tokenize(s: str) -> List[Tok]:\n    toks: List[Tok] = []\n    for m in TOKEN_RE.finditer(s):\n        typ = m.lastgroup\n        val = m.group()\n        if typ == "SKIP":\n            continue\n        if typ == "MISMATCH":\n            raise ValueError(f"Лексическая ошибка: неожиданный символ {val!r}")\n        toks.append(Tok(typ, val))\n    return toks\n\n# AST\n@dataclass(frozen=True)\nclass Node: ...\n\n@dataclass(frozen=True)\nclass Var(Node):\n    name: str\n\n@dataclass(frozen=True)\nclass Not(Node):\n    x: Node\n\n@dataclass(frozen=True)\nclass Bin(Node):\n    op: str  # \'AND\',\'OR\',\'IMP\'\n    a: Node\n    b: Node\n\nclass Parser:\n    # приоритеты: !, *, +, ->\n    def __init__(self, toks: List[Tok]):\n        self.toks = toks\n        self.i = 0\n\n    #посмотреть текущий токен\n    def peek(self) -> Optional[Tok]:\n        return self.toks[self.i] if self.i < len(self.toks) else None\n\n    #проверка на ожидаемый тип\n    def eat(self, typ: str) -> Tok:\n        t = self.peek()\n        if not t or t.typ != typ:\n            got = t.typ if t else "EOF"\n            raise ValueError(f"Синтаксическая ошибка: ожидалось {typ}, получено {got}")\n        self.i += 1\n        return t\n\n    def parse(self) -> Node:\n        node = self.parse_imp()\n        if self.peek() is not None:\n            raise ValueError(f"Синтаксическая ошибка: лишний токен {self.peek().val!r}")\n        return node\n\n    #импликация\n    def parse_imp(self) -> Node:\n        left = self.parse_or()\n        t = self.peek()\n        if t and t.typ == "ARROW":\n            self.eat("ARROW")\n            right = self.parse_imp()\n            return Bin("IMP", left, right)\n        return left\n\n    #дизъюнкция\n    def parse_or(self) -> Node:\n        node = self.parse_and()\n        while True:\n            t = self.peek()\n            if t and t.typ == "OR":\n                self.eat("OR")\n                node = Bin("OR", node, self.parse_and())\n            else:\n                break\n        return node\n\n    #конъюнкция\n    def parse_and(self) -> Node:\n        node = self.parse_not()\n        while True:\n            t = self.peek()\n            if t and t.typ == "AND":\n                self.eat("AND")\n                node = Bin("AND", node, self.parse_not())\n            else:\n                break\n        return node\n\n    #отрицание\n    def parse_not(self) -> Node:\n        t = self.peek()\n        if t and t.typ == "NOT":\n            self.eat("NOT")\n            return Not(self.parse_not())\n        return self.parse_atom()\n\n    def parse_atom(self) -> Node:\n        t = self.peek()\n        if not t:\n            raise ValueError("Синтаксическая ошибка: неожиданный конец ввода")\n        if t.typ == "ID":\n            self.eat("ID")\n            return Var(t.val)\n        if t.typ == "LP":\n            self.eat("LP")\n            node = self.parse_imp()\n            self.eat("RP")\n            return node\n        raise ValueError(f"Синтаксическая ошибка: неожиданный токен {t.val!r}")\n\n#выражение Вход - формула (текст) Выход - дерево\ndef parse_expr(expr_text: str) -> Node:\n    toks = tokenize(expr_text)\n    return Parser(toks).parse()\n\n#Сбор переменных\ndef collect_vars(node: Node, out: Set[str]) -> None:\n    if isinstance(node, Var):\n        out.add(node.name)\n    elif isinstance(node, Not):\n        collect_vars(node.x, out)\n    elif isinstance(node, Bin):\n        collect_vars(node.a, out); collect_vars(node.b, out)\n\n#Вычисление формулы\ndef eval_node(node: Node, env: Dict[str, bool]) -> bool:\n    if isinstance(node, Var):\n        return bool(env.get(node.name, False))\n    if isinstance(node, Not):\n        return not eval_node(node.x, env)\n    if isinstance(node, Bin):\n        if node.op == "AND":\n            return eval_node(node.a, env) and eval_node(node.b, env)\n        if node.op == "OR":\n            return eval_node(node.a, env) or eval_node(node.b, env)\n        # -> = ! +\n        if node.op == "IMP":\n            return (not eval_node(node.a, env)) or eval_node(node.b, env)\n    raise TypeError("Неизвестный тип узла AST")\n\n#таблица истинности\ndef truth_table(expr_text: str) -> Tuple[List[str], List[Tuple[Dict[str, bool], bool]]]:\n    ast = parse_expr(expr_text)\n    vs: Set[str] = set()\n    collect_vars(ast, vs)\n    vars_ = sorted(vs)\n    rows = []\n    for bits in itertools.product([False, True], repeat=len(vars_)):\n        env = dict(zip(vars_, bits))\n        rows.append((env, eval_node(ast, env)))\n    return vars_, rows\n\ndef is_tautology(expr_text: str) -> bool:\n    _, rows = truth_table(expr_text)\n    return all(out for _, out in rows)\n\ndef is_satisfiable(expr_text: str) -> bool:\n    _, rows = truth_table(expr_text)\n    return any(out for _, out in rows)\n\n#База знаний (Хорн)\n@dataclass\nclass HornRule:\n    rid: int\n    original_text: str\n    antecedents: List[str]      # условия слева (and)\n    consequent: str             # один вывод\n\n    def cond_text(self) -> str:\n        return "*".join(self.antecedents) if self.antecedents else ""\n\nclass MiniLogicKB:\n    def __init__(self):\n        self.facts: Set[str] = set() #A\n        self.rules: List[HornRule] = [] #Ak\n        self._next_rid = 1\n\n    #очистить все правила и факты\n    def clear(self):\n        self.facts.clear()\n        self.rules.clear()\n        self._next_rid = 1\n\n    # добавить факты\n    def alp(self, arg: str):\n        arg = arg.strip()\n        if arg.startswith("[") and arg.endswith("]"):\n            inner = arg[1:-1].strip()\n            if not inner:\n                return\n            items = [x.strip() for x in inner.split(",")]\n            for it in items:\n                if it:\n                    self._add_fact(it)\n        else:\n            self._add_fact(arg)\n\n    def _add_fact(self, name: str):\n        if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", name):\n            raise ValueError(f"Некорректное имя факта: {name}")\n        self.facts.add(name)\n\n    #добавить аксиому\n    def axi(self, expr_text: str):\n        """\n        Разрешается форму:\n          a*b -> c+d+e\n        Интерпретация как сокращение:\n          (a*b -> c), (a*b -> d), (a*b -> e)\n        """\n        expr_text = expr_text.strip()\n        # Требуем импликацию на верхнем уровне\n        ast = parse_expr(expr_text)\n        if not (isinstance(ast, Bin) and ast.op == "IMP"):\n            raise ValueError("axi(...) ожидает импликацию вида X -> Y")\n\n        ants = self._extract_conj_atoms(ast.a)\n        cons = self._extract_disj_atoms(ast.b)\n\n        rid = self._next_rid\n        self._next_rid += 1\n\n        for c in cons:\n            self.rules.append(HornRule(rid=rid, original_text=expr_text, antecedents=ants, consequent=c))\n\n    def _extract_conj_atoms(self, node: Node) -> List[str]:\n        # Разрешаем только конъюнкцию атомов (Var)\n        atoms: List[str] = []\n        def walk(n: Node):\n            if isinstance(n, Var):\n                atoms.append(n.name)\n                return\n            if isinstance(n, Bin) and n.op == "AND":\n                walk(n.a); walk(n.b); return\n            raise ValueError("В левой части правила допустимы только атомы, соединённые * (И)")\n        walk(node)\n        # Удалим дубли, сохранив порядок\n        seen = set()\n        out = []\n        for a in atoms:\n            if a not in seen:\n                seen.add(a); out.append(a)\n        return out\n\n    def _extract_disj_atoms(self, node: Node) -> List[str]:\n        # Разрешаем только дизъюнкцию атомов (Var)\n        atoms: List[str] = []\n        def walk(n: Node):\n            if isinstance(n, Var):\n                atoms.append(n.name); return\n            if isinstance(n, Bin) and n.op == "OR":\n                walk(n.a); walk(n.b); return\n            raise ValueError("В правой части правила допустимы только атомы, соединённые + (ИЛИ)")\n        walk(node)\n        # Удалим дубли, сохранив порядок\n        seen = set()\n        out = []\n        for a in atoms:\n            if a not in seen:\n                seen.add(a); out.append(a)\n        return out\n\n    def list_alp(self):\n        print("A = { " + ", ".join(sorted(self.facts)) + " }")\n\n    def list_axi(self):\n        if not self.rules:\n            print("(правил нет)")\n            return\n        for r in self.rules:\n            left = "*".join(r.antecedents)\n            print(f"   axi#{r.rid}: {r.original_text}  ::  {left} -> {r.consequent}")\n\n    #вывод (правило P)\n    def theorem(self, goal_text: str) -> bool:\n        goal_text = goal_text.strip()\n        goal_atoms = self._parse_goal_conj(goal_text)\n\n        #все что уже известно как истинное\n        derived: Set[str] = set(self.facts)\n        derived_order: List[str] = []\n        why: Dict[str, Tuple[Optional[HornRule], List[str]]] = {}\n\n        for f in sorted(self.facts):\n            derived_order.append(f)\n            why[f] = (None, [])\n\n        #прямой вывод\n        changed = True\n        while changed:\n            changed = False\n            for rule in self.rules:\n                if rule.consequent in derived:\n                    continue\n                if all(a in derived for a in rule.antecedents):\n                    derived.add(rule.consequent)\n                    derived_order.append(rule.consequent)\n                    why[rule.consequent] = (rule, list(rule.antecedents))\n                    changed = True\n        #доказано если все переменные есть в derived\n        ok = all(g in derived for g in goal_atoms)\n        self._print_trace(goal_atoms, ok, derived_order, why)\n        return ok\n\n    def _parse_goal_conj(self, text: str) -> List[str]:\n        # goal: "a*b*c" (только конъюнкция атомов)\n        text = text.strip()\n        if not text:\n            raise ValueError("Пустая цель theorem(...)")\n        parts = [p.strip() for p in text.split("*")]\n        if not parts or any(not p for p in parts):\n            raise ValueError("Цель theorem(...) должна быть конъюнкцией атомов через *")\n        for p in parts:\n            if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", p):\n                raise ValueError(f"Некорректный атом в цели: {p}")\n        # уникализируем, сохраняя порядок\n        seen=set(); out=[]\n        for p in parts:\n            if p not in seen:\n                seen.add(p); out.append(p)\n        return out\n\n    # Упрощённая сигнатура\n    def _print_trace(self, goal_atoms, ok, derived, why):\n        print("\\n=== Трассировка вывода ===")\n        print("Факты (A): " + (", ".join(sorted(self.facts)) if self.facts else "(нет)"))\n        print("Правила (Ak):")\n        if not self.rules:\n            print("  (нет)")\n        else:\n            for r in self.rules:\n                left = "*".join(r.antecedents)\n                print(f"  axi#{r.rid}: {r.original_text}: {left} -> {r.consequent}")\n\n        print("Выведено:")\n        for x in derived:\n            rule, conds = why.get(x, (None, []))\n            if rule is None:\n                print(f"  {x}  (дано)")\n            else:\n                cond_text = " & ".join(conds) if conds else ""\n                print(f"  {x}  (из axi#{rule.rid}: {rule.original_text} при {cond_text})")\n\n        print("Цель: " + " * ".join(goal_atoms))\n        print("Результат: " + ("ИСТИНА" if ok else "ЛОЖЬ"))\n        print("==========================\\n")\n\n#Команды\nHELP_TEXT = """MiniLogic — справка.\n\nСинтаксис выражений:\n  Переменные: a, b, x1, Has_Passport\n  Операции:  ! (НЕ), * (И), + (ИЛИ), -> (ИМПЛИКАЦИЯ), скобки ( )\n\nБаза знаний:\n  alp(name)                — добавить факт (элемент алфавита A)\n  alp([a, b, c])           — добавить сразу несколько фактов\n  axi(expr)                — добавить аксиому(ы) в виде Хорновых правил\n                             Пример: axi(a*e -> b+c+d)  ⇒ (a*e→b), (a*e→c), (a*e→d)\n  list_alp()               — показать все факты\n  list_axi()               — показать все правила\n  clear()                  — очистить базу знаний\n\nВывод (P):\n  theorem(goal)            — проверить выводимость конъюнкции атомов\n                             Пример: theorem(b*c)\n\nБулева алгебра и таблицы истинности:\n  truth(expr)              — вывести таблицу истинности\n  equiv(e1, e2)            — проверить логическую эквивалентность\n  laws()                   — проверить набор классических тождеств\n  sat(expr)                — выполнима ли формула (есть ли строка, где она истинна)\n  valid(expr)              — тавтология ли формула (истинна ли на всех строках)\n\nПрочее:\n  help()                   — эта справка\n  quit / exit              — выход\n"""\n\n#вывод таблицы истинности\ndef cmd_truth(expr: str):\n    vars_, rows = truth_table(expr)\n    if not vars_:\n        # константа\n        out = rows[0][1]\n        print(f"F\\n-\\n{1 if out else 0}")\n        return\n    print(" | ".join(vars_) + " | F")\n    print("-" * (4 * len(vars_) + 3))\n    for env, out in rows:\n        line = " | ".join("1" if env[v] else "0" for v in vars_) + " | " + ("1" if out else "0")\n        print(line)\n\n#проверка эквивалентности\ndef cmd_equiv(arg: str):\n    parts = [p.strip() for p in arg.split(",")]\n    if len(parts) != 2:\n        raise ValueError("equiv(e1, e2) ожидает два выражения через запятую")\n    e1, e2 = parts\n    vars1, rows1 = truth_table(e1)\n    vars2, rows2 = truth_table(e2)\n    vars_all = sorted(set(vars1) | set(vars2))\n\n    def eval_on(expr: str, env: Dict[str, bool]) -> bool:\n        return eval_node(parse_expr(expr), env)\n\n    ok = True\n    for bits in itertools.product([False, True], repeat=len(vars_all)):\n        env = dict(zip(vars_all, bits))\n        if eval_on(e1, env) != eval_on(e2, env):\n            ok = False\n            break\n    print("Эквивалентны: " + ("ДА" if ok else "НЕТ"))\n\n#проверка тождеств\ndef cmd_laws():\n    # набор классических тождеств\n    laws = [\n        ("коммутативность_and", "a*b", "b*a"),\n        ("коммутативность_or",  "a+b", "b+a"),\n        ("дистрибутивность1",   "a*(b+c)", "a*b + a*c"),\n        ("дистрибутивность2",   "a+b*c", "(a+b)*(a+c)"),\n        ("идемпотентность_and", "a*a", "a"),\n        ("идемпотентность_or",  "a+a", "a"),\n        ("де_морган1",          "!(a*b)", "!a + !b"),\n        ("де_морган2",          "!(a+b)", "!a * !b"),\n        ("импликация",          "a->b", "!a + b"),\n    ]\n    print("Проверка тождеств булевой алгебры:")\n    for name, e1, e2 in laws:\n        # быстро: проверка эквивалентности\n        vars_all = sorted(set(collect(e1)) | set(collect(e2)))\n        ok = True\n        for bits in itertools.product([False, True], repeat=len(vars_all)):\n            env = dict(zip(vars_all, bits))\n            if eval_node(parse_expr(e1), env) != eval_node(parse_expr(e2), env):\n                ok = False\n                break\n        print(f"  {name:20s}: {\'OK\' if ok else \'FAIL\'}   ({e1} ≡ {e2})")\n\ndef collect(expr: str) -> Set[str]:\n    ast = parse_expr(expr)\n    s: Set[str] = set()\n    collect_vars(ast, s)\n    return s\n\n#выполнимость\ndef cmd_sat(expr: str):\n    print("Выполнима: " + ("ДА" if is_satisfiable(expr) else "НЕТ"))\n\n#тавтолгогия\ndef cmd_valid(expr: str):\n    print("Тавтология: " + ("ДА" if is_tautology(expr) else "НЕТ"))\n\n#разбирает строку на имя команды и аргумент\ndef parse_command(line: str) -> Tuple[str, str]:\n    """\n    Поддержка:\n      name\n      name()\n      name(arg)\n    """\n    line = line.strip()\n    # name(...) ?\n    m = re.match(r"^([A-Za-z_][A-Za-z0-9_]*)\\s*(\\((.*)\\))?$", line)\n    if not m:\n        raise ValueError("Не распознан ввод. Введите help() для справки.")\n    name = m.group(1)\n    has_parens = m.group(2) is not None\n    arg = (m.group(3) or "").strip()\n    # если без скобок, arg пуст\n    if not has_parens:\n        arg = ""\n    return name, arg\n\ndef run_script(path: str):\n    kb = MiniLogicKB()\n\n    def help_():\n        print(HELP_TEXT)\n\n    # диспетчер\n    def dispatch(name: str, arg: str):\n        if name in ("quit", "exit"):\n            raise SystemExit\n\n        if name == "help":\n            help_(); return\n        if name == "clear":\n            kb.clear(); print("Очищено."); return\n        if name == "alp":\n            kb.alp(arg); return\n        if name == "axi":\n            kb.axi(arg); return\n        if name == "list_alp":\n            kb.list_alp(); return\n        if name == "list_axi":\n            kb.list_axi(); return\n        if name == "theorem":\n            kb.theorem(arg); return\n        if name == "truth":\n            cmd_truth(arg); return\n        if name == "equiv":\n            cmd_equiv(arg); return\n        if name == "laws":\n            cmd_laws(); return\n        if name == "sat":\n            cmd_sat(arg); return\n        if name == "valid":\n            cmd_valid(arg); return\n\n        raise ValueError(f"Неизвестная функция: {name}")\n\n    # чтение файла\n    with open(path, "r", encoding="utf-8") as f:\n        for raw in f:\n            line = raw.strip()\n            if not line:\n                continue\n            # убираем комментарии\n            if "#" in line:\n                line = line.split("#", 1)[0].strip()\n                if not line:\n                    continue\n            print(f">> {line}")\n            try:\n                name, arg = parse_command(line)\n                # разрешаем формы без аргументов без скобок: help / list_alp / list_axi / laws / clear\n                if name in {"help", "list_alp", "list_axi", "laws", "clear"} and arg == "" and "(" not in line:\n                    dispatch(name, "")\n                else:\n                    dispatch(name, arg)\n            except SystemExit:\n                return\n            except Exception as e:\n                print(f"Ошибка: {e}")\n\ndef main(argv: List[str]) -> int:\n    if len(argv) != 2:\n        print("Использование: python minilogic.py <script.txt>")\n        return 2\n    run_script(argv[1])\n    return 0\n\nif __name__ == "__main__":\n    raise SystemExit(main(sys.argv))\n', encoding="utf-8")
print('Сохранён minilogic.py')


## Сценарий MiniLogic


In [ ]:
from pathlib import Path
Path('scenario_flight_simple.txt').write_text('help\nclear\n\n# СИТУАЦИЯ 1 (норм): всё есть -> можно лететь международно (ИСТИНА)\nalp([has_passport, has_ticket, has_visa])\naxi(has_passport*has_ticket -> can_board)\naxi(can_board*has_visa -> can_fly_international)\nlist_alp\nlist_axi\ntheorem(can_fly_international)\n\n# СИТУАЦИЯ 2 (проблема): нет визы -> международный перелёт не выводится (ЛОЖЬ)\nclear\nalp([has_passport, has_ticket])\naxi(has_passport*has_ticket -> can_board)\naxi(can_board*has_visa -> can_fly_international)\nlist_alp\nlist_axi\ntheorem(can_fly_international)\n\n# СИТУАЦИЯ 3 (проблема): нет билета -> даже посадка НЕ выводится (ЛОЖЬ)\nclear\nalp([has_passport, has_visa])\naxi(has_passport*has_ticket -> can_board)\naxi(can_board*has_visa -> can_fly_international)\nlist_alp\nlist_axi\ntheorem(can_board)\ntheorem(can_fly_international)\n', encoding="utf-8")
print('Сохранён scenario_flight_simple.txt')


## Запуск MiniLogic


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, 'minilogic.py', 'scenario_flight_simple.txt'], check=True)


## Модуль MiniProlog


In [ ]:
from pathlib import Path
Path('miniprolog.py').write_text('# -*- coding: utf-8 -*-\n"""\nMiniProlog — логическое программирование / исчисление предикатов:\n- термы: Const, Var, Func\n- предикаты/атомы: p(t1,...,tn)\n- клаузы: факт (head.) и правило (head :- body1, body2.)\n- вывод: SLD-резолюция (backward chaining) + унификация + подстановки\n- трассировка шагов вывода\n"""\n\nimport sys\nimport re\nimport itertools\nfrom dataclasses import dataclass\nfrom typing import List, Tuple, Dict, Optional, Set, Iterable\n\n# 1) Булева алгебра (из ПР1): !, *, +, ->, truth/equiv/laws\n\nTOKEN_SPEC = [\n    ("SKIP",   r"[ \\t\\r\\n]+"),\n    ("ARROW",  r"->"),\n    ("NOT",    r"!"),\n    ("AND",    r"\\*"),\n    ("OR",     r"\\+"),\n    ("LP",     r"\\("),\n    ("RP",     r"\\)"),\n    ("ID",     r"[A-Za-z_][A-Za-z0-9_]*"),\n    ("MISMATCH", r"."),\n]\nTOKEN_RE = re.compile("|".join(f"(?P<{name}>{pattern})" for name, pattern in TOKEN_SPEC))\n\n@dataclass(frozen=True)\nclass Tok:\n    typ: str\n    val: str\n\ndef tokenize_bool(s: str) -> List[Tok]:\n    toks: List[Tok] = []\n    for m in TOKEN_RE.finditer(s):\n        typ = m.lastgroup\n        val = m.group()\n        if typ == "SKIP":\n            continue\n        if typ == "MISMATCH":\n            raise ValueError(f"Лексическая ошибка: неожиданный символ {val!r}")\n        toks.append(Tok(typ, val))\n    return toks\n\n@dataclass(frozen=True)\nclass Node: ...\n\n@dataclass(frozen=True)\nclass VarB(Node):\n    name: str\n\n@dataclass(frozen=True)\nclass NotB(Node):\n    x: Node\n\n@dataclass(frozen=True)\nclass BinB(Node):\n    op: str  # AND/OR/IMP\n    a: Node\n    b: Node\n\nclass ParserB:\n    # ! * + ->\n    def __init__(self, toks: List[Tok]):\n        self.toks = toks\n        self.i = 0\n\n    def peek(self) -> Optional[Tok]:\n        return self.toks[self.i] if self.i < len(self.toks) else None\n\n    def eat(self, typ: str) -> Tok:\n        t = self.peek()\n        if not t or t.typ != typ:\n            got = t.typ if t else "EOF"\n            raise ValueError(f"Синтаксическая ошибка: ожидалось {typ}, получено {got}")\n        self.i += 1\n        return t\n\n    def parse(self) -> Node:\n        node = self.parse_imp()\n        if self.peek() is not None:\n            raise ValueError(f"Синтаксическая ошибка: лишний токен {self.peek().val!r}")\n        return node\n\n    def parse_imp(self) -> Node:\n        left = self.parse_or()\n        t = self.peek()\n        if t and t.typ == "ARROW":\n            self.eat("ARROW")\n            right = self.parse_imp()\n            return BinB("IMP", left, right)\n        return left\n\n    def parse_or(self) -> Node:\n        node = self.parse_and()\n        while True:\n            t = self.peek()\n            if t and t.typ == "OR":\n                self.eat("OR")\n                node = BinB("OR", node, self.parse_and())\n            else:\n                break\n        return node\n\n    def parse_and(self) -> Node:\n        node = self.parse_not()\n        while True:\n            t = self.peek()\n            if t and t.typ == "AND":\n                self.eat("AND")\n                node = BinB("AND", node, self.parse_not())\n            else:\n                break\n        return node\n\n    def parse_not(self) -> Node:\n        t = self.peek()\n        if t and t.typ == "NOT":\n            self.eat("NOT")\n            return NotB(self.parse_not())\n        return self.parse_atom()\n\n    def parse_atom(self) -> Node:\n        t = self.peek()\n        if not t:\n            raise ValueError("Синтаксическая ошибка: неожиданный конец ввода")\n        if t.typ == "ID":\n            self.eat("ID")\n            return VarB(t.val)\n        if t.typ == "LP":\n            self.eat("LP")\n            node = self.parse_imp()\n            self.eat("RP")\n            return node\n        raise ValueError(f"Синтаксическая ошибка: неожиданный токен {t.val!r}")\n\ndef parse_bool(expr_text: str) -> Node:\n    return ParserB(tokenize_bool(expr_text)).parse()\n\ndef collect_vars_bool(node: Node, out: Set[str]) -> None:\n    if isinstance(node, VarB):\n        out.add(node.name)\n    elif isinstance(node, NotB):\n        collect_vars_bool(node.x, out)\n    elif isinstance(node, BinB):\n        collect_vars_bool(node.a, out); collect_vars_bool(node.b, out)\n\ndef eval_bool(node: Node, env: Dict[str, bool]) -> bool:\n    if isinstance(node, VarB):\n        return bool(env.get(node.name, False))\n    if isinstance(node, NotB):\n        return not eval_bool(node.x, env)\n    if isinstance(node, BinB):\n        if node.op == "AND":\n            return eval_bool(node.a, env) and eval_bool(node.b, env)\n        if node.op == "OR":\n            return eval_bool(node.a, env) or eval_bool(node.b, env)\n        if node.op == "IMP":\n            return (not eval_bool(node.a, env)) or eval_bool(node.b, env)\n    raise TypeError("Неизвестный узел AST")\n\ndef truth_table(expr_text: str):\n    ast = parse_bool(expr_text)\n    vs: Set[str] = set()\n    collect_vars_bool(ast, vs)\n    vars_ = sorted(vs)\n    rows = []\n    for bits in itertools.product([False, True], repeat=len(vars_)):\n        env = dict(zip(vars_, bits))\n        rows.append((env, eval_bool(ast, env)))\n    return vars_, rows\n\ndef is_tautology(expr_text: str) -> bool:\n    _, rows = truth_table(expr_text)\n    return all(out for _, out in rows)\n\ndef is_satisfiable(expr_text: str) -> bool:\n    _, rows = truth_table(expr_text)\n    return any(out for _, out in rows)\n\ndef cmd_truth(expr: str):\n    vars_, rows = truth_table(expr)\n    if not vars_:\n        out = rows[0][1]\n        print(f"F\\n-\\n{1 if out else 0}")\n        return\n    print(" | ".join(vars_) + " | F")\n    print("-" * (4 * len(vars_) + 3))\n    for env, out in rows:\n        line = " | ".join("1" if env[v] else "0" for v in vars_) + " | " + ("1" if out else "0")\n        print(line)\n\ndef cmd_equiv(arg: str):\n    parts = [p.strip() for p in arg.split(",")]\n    if len(parts) != 2:\n        raise ValueError("equiv(e1, e2) ожидает два выражения через запятую")\n    e1, e2 = parts\n    vars1, _ = truth_table(e1)\n    vars2, _ = truth_table(e2)\n    vars_all = sorted(set(vars1) | set(vars2))\n\n    ok = True\n    for bits in itertools.product([False, True], repeat=len(vars_all)):\n        env = dict(zip(vars_all, bits))\n        if eval_bool(parse_bool(e1), env) != eval_bool(parse_bool(e2), env):\n            ok = False\n            break\n    print("Эквивалентны: " + ("ДА" if ok else "НЕТ"))\n\ndef cmd_laws():\n    laws = [\n        ("коммутативность_and", "a*b", "b*a"),\n        ("коммутативность_or",  "a+b", "b+a"),\n        ("дистрибутивность1",   "a*(b+c)", "a*b + a*c"),\n        ("дистрибутивность2",   "a+b*c", "(a+b)*(a+c)"),\n        ("идемпотентность_and", "a*a", "a"),\n        ("идемпотентность_or",  "a+a", "a"),\n        ("де_морган1",          "!(a*b)", "!a + !b"),\n        ("де_морган2",          "!(a+b)", "!a * !b"),\n        ("импликация",          "a->b", "!a + b"),\n    ]\n    print("Проверка тождеств булевой алгебры:")\n    for name, e1, e2 in laws:\n        vars_all = sorted(set(v for v,_ in []) | set())  # заглушка, ниже реальный сбор\n        vs = set(); collect_vars_bool(parse_bool(e1), vs); collect_vars_bool(parse_bool(e2), vs)\n        vars_all = sorted(vs)\n\n        ok = True\n        for bits in itertools.product([False, True], repeat=len(vars_all)):\n            env = dict(zip(vars_all, bits))\n            if eval_bool(parse_bool(e1), env) != eval_bool(parse_bool(e2), env):\n                ok = False\n                break\n        print(f"  {name:20s}: {\'OK\' if ok else \'FAIL\'}   ({e1} ≡ {e2})")\n\ndef cmd_sat(expr: str):\n    print("Выполнима: " + ("ДА" if is_satisfiable(expr) else "НЕТ"))\n\ndef cmd_valid(expr: str):\n    print("Тавтология: " + ("ДА" if is_tautology(expr) else "НЕТ"))\n\n\n# 2) Исчисление предикатов: термы, атомы, клаузы, унификация\n\n# Термы\n@dataclass(frozen=True)\nclass Term: ...\n\n@dataclass(frozen=True)\nclass Var(Term):\n    name: str  # начинаем с заглавной буквы или \'_\' (как в Prolog)\n\n@dataclass(frozen=True)\nclass Const(Term):\n    name: str  # обычная константа/атом (нижний регистр)\n\n@dataclass(frozen=True)\nclass Func(Term):\n    name: str\n    args: Tuple[Term, ...]\n\n@dataclass(frozen=True)\nclass Atom:\n    pred: str\n    args: Tuple[Term, ...]\n\n@dataclass\nclass Clause:\n    cid: int\n    head: Atom\n    body: Tuple[Atom, ...]   # пусто = факт\n    text: str\n\n# Лексер/парсер для предикатов\n\nP_TOKEN = re.compile(r"""\n    (?P<SKIP>\\s+)\n  | (?P<COLON_DASH>:-)\n  | (?P<LP>\\()\n  | (?P<RP>\\))\n  | (?P<COMMA>,)\n  | (?P<DOT>\\.)\n  | (?P<ID>[A-Za-z_][A-Za-z0-9_]*)\n  | (?P<MISMATCH>.)\n""", re.VERBOSE)\n\n@dataclass(frozen=True)\nclass PTok:\n    typ: str\n    val: str\n\ndef p_tokenize(s: str) -> List[PTok]:\n    out: List[PTok] = []\n    for m in P_TOKEN.finditer(s):\n        typ = m.lastgroup\n        val = m.group()\n        if typ == "SKIP":\n            continue\n        if typ == "MISMATCH":\n            raise ValueError(f"Лексическая ошибка (предикаты): {val!r}")\n        out.append(PTok(typ, val))\n    return out\n\nclass PParser:\n    def __init__(self, toks: List[PTok]):\n        self.toks = toks\n        self.i = 0\n\n    def peek(self) -> Optional[PTok]:\n        return self.toks[self.i] if self.i < len(self.toks) else None\n\n    def eat(self, typ: str) -> PTok:\n        t = self.peek()\n        if not t or t.typ != typ:\n            got = t.typ if t else "EOF"\n            raise ValueError(f"Синтаксическая ошибка (предикаты): ожидалось {typ}, получено {got}")\n        self.i += 1\n        return t\n\n    def parse_term(self) -> Term:\n        t = self.peek()\n        if not t or t.typ != "ID":\n            raise ValueError("Ожидался терм")\n        name = self.eat("ID").val\n        # переменная: начинается с заглавной или \'_\'\n        if name[0].isupper() or name[0] == "_":\n            return Var(name)\n\n        # функция/структура?\n        if self.peek() and self.peek().typ == "LP":\n            self.eat("LP")\n            args: List[Term] = []\n            if self.peek() and self.peek().typ != "RP":\n                args.append(self.parse_term())\n                while self.peek() and self.peek().typ == "COMMA":\n                    self.eat("COMMA")\n                    args.append(self.parse_term())\n            self.eat("RP")\n            return Func(name, tuple(args))\n\n        return Const(name)\n\n    def parse_atom(self) -> Atom:\n        t = self.peek()\n        if not t or t.typ != "ID":\n            raise ValueError("Ожидался предикат")\n        pred = self.eat("ID").val\n\n        if not (self.peek() and self.peek().typ == "LP"):\n            return Atom(pred, tuple())\n\n        self.eat("LP")\n        args: List[Term] = []\n        if self.peek() and self.peek().typ != "RP":\n            args.append(self.parse_term())\n            while self.peek() and self.peek().typ == "COMMA":\n                self.eat("COMMA")\n                args.append(self.parse_term())\n        self.eat("RP")\n        return Atom(pred, tuple(args))\n\n    def parse_clause(self) -> Tuple[Atom, Tuple[Atom, ...]]:\n        head = self.parse_atom()\n        if self.peek() and self.peek().typ == "COLON_DASH":\n            self.eat("COLON_DASH")\n            body: List[Atom] = []\n            body.append(self.parse_atom())\n            while self.peek() and self.peek().typ == "COMMA":\n                self.eat("COMMA")\n                body.append(self.parse_atom())\n            return head, tuple(body)\n        return head, tuple()\n\ndef parse_predicate_clause(text: str) -> Tuple[Atom, Tuple[Atom, ...]]:\n    # уберём завершающую точку, если есть\n    s = text.strip()\n    if s.endswith("."):\n        s = s[:-1].strip()\n    pp = PParser(p_tokenize(s))\n    head, body = pp.parse_clause()\n    if pp.peek() is not None:\n        raise ValueError(f"Лишние токены в конце: {pp.peek().val!r}")\n    return head, body\n\ndef parse_query_atoms(text: str) -> Tuple[Atom, ...]:\n    # запрос допускаем как список атомов через запятую: a(...), b(...)\n    s = text.strip()\n    if s.endswith("."):\n        s = s[:-1].strip()\n    toks = p_tokenize(s)\n    pp = PParser(toks)\n    atoms: List[Atom] = []\n    atoms.append(pp.parse_atom())\n    while pp.peek() and pp.peek().typ == "COMMA":\n        pp.eat("COMMA")\n        atoms.append(pp.parse_atom())\n    if pp.peek() is not None:\n        raise ValueError(f"Лишние токены в запросе: {pp.peek().val!r}")\n    return tuple(atoms)\n\n# Унификация / подстановка\n\nSubst = Dict[str, Term]  # Var.name -> Term\n\ndef deref(t: Term, s: Subst) -> Term:\n    # найти конечное значение переменной\n    while isinstance(t, Var) and t.name in s:\n        t = s[t.name]\n    return t\n\ndef occurs(v: str, t: Term, s: Subst) -> bool:\n    t = deref(t, s)\n    if isinstance(t, Var):\n        return t.name == v\n    if isinstance(t, Func):\n        return any(occurs(v, a, s) for a in t.args)\n    return False\n\ndef unify(t1: Term, t2: Term, s: Subst) -> Optional[Subst]:\n    t1 = deref(t1, s)\n    t2 = deref(t2, s)\n\n    if t1 == t2:\n        return s\n\n    if isinstance(t1, Var):\n        # occurs-check можно оставить (без него иногда проще, но риск циклов)\n        if occurs(t1.name, t2, s):\n            return None\n        s2 = dict(s); s2[t1.name] = t2\n        return s2\n\n    if isinstance(t2, Var):\n        if occurs(t2.name, t1, s):\n            return None\n        s2 = dict(s); s2[t2.name] = t1\n        return s2\n\n    if isinstance(t1, Const) and isinstance(t2, Const):\n        return s if t1.name == t2.name else None\n\n    if isinstance(t1, Func) and isinstance(t2, Func):\n        if t1.name != t2.name or len(t1.args) != len(t2.args):\n            return None\n        s2 = dict(s)\n        for a, b in zip(t1.args, t2.args):\n            s2 = unify(a, b, s2)\n            if s2 is None:\n                return None\n        return s2\n\n    return None\n\ndef unify_atom(a: Atom, b: Atom, s: Subst) -> Optional[Subst]:\n    if a.pred != b.pred or len(a.args) != len(b.args):\n        return None\n    s2 = dict(s)\n    for x, y in zip(a.args, b.args):\n        s2 = unify(x, y, s2)\n        if s2 is None:\n            return None\n    return s2\n\ndef apply_subst_term(t: Term, s: Subst) -> Term:\n    t = deref(t, s)\n    if isinstance(t, Var):\n        return t\n    if isinstance(t, Const):\n        return t\n    if isinstance(t, Func):\n        return Func(t.name, tuple(apply_subst_term(a, s) for a in t.args))\n    return t\n\ndef apply_subst_atom(a: Atom, s: Subst) -> Atom:\n    return Atom(a.pred, tuple(apply_subst_term(t, s) for t in a.args))\n\ndef vars_in_atom(a: Atom) -> Set[str]:\n    out: Set[str] = set()\n    def walk(t: Term):\n        t = deref(t, {})\n        if isinstance(t, Var):\n            out.add(t.name)\n        elif isinstance(t, Func):\n            for x in t.args:\n                walk(x)\n    for arg in a.args:\n        walk(arg)\n    return out\n\n#  Стандартизация переменных (снятие кванторов)\n\ndef rename_term(t: Term, mapping: Dict[str, str]) -> Term:\n    if isinstance(t, Var):\n        return Var(mapping.get(t.name, t.name))\n    if isinstance(t, Const):\n        return t\n    if isinstance(t, Func):\n        return Func(t.name, tuple(rename_term(a, mapping) for a in t.args))\n    return t\n\ndef rename_atom(a: Atom, mapping: Dict[str, str]) -> Atom:\n    return Atom(a.pred, tuple(rename_term(t, mapping) for t in a.args))\n\ndef standardize_apart(cl: Clause, fresh_id: int) -> Clause:\n    # переименуем все переменные в клаузе: X -> X__fresh_id\n    all_vars: Set[str] = set()\n    all_vars |= vars_in_atom(cl.head)\n    for b in cl.body:\n        all_vars |= vars_in_atom(b)\n    mapping = {v: f"{v}__{fresh_id}" for v in all_vars}\n    head2 = rename_atom(cl.head, mapping)\n    body2 = tuple(rename_atom(b, mapping) for b in cl.body)\n    return Clause(cid=cl.cid, head=head2, body=body2, text=cl.text)\n\n#  База знаний + SLD-резолюция\n\nclass MiniPrologKB:\n    def __init__(self):\n        self.clauses: List[Clause] = []\n        self._next_cid = 1\n        self._fresh = 1\n\n    def clear(self):\n        self.clauses.clear()\n        self._next_cid = 1\n        self._fresh = 1\n\n    def add_fact(self, atom_text: str):\n        head, body = parse_predicate_clause(atom_text)\n        if body:\n            raise ValueError("fact(...) ожидает факт без \':-\'")\n        self.clauses.append(Clause(self._next_cid, head, tuple(), atom_text.strip()))\n        self._next_cid += 1\n\n    def add_rule(self, rule_text: str):\n        head, body = parse_predicate_clause(rule_text)\n        if not body:\n            raise ValueError("rule(...) ожидает \'head :- body1, body2\'")\n        self.clauses.append(Clause(self._next_cid, head, body, rule_text.strip()))\n        self._next_cid += 1\n\n    def axi(self, text: str):\n        # поддержим: head :- body...   ИЛИ   body -> head (с \'*\' как конъюнкция)\n        s = text.strip()\n        if ":-" in s:\n            return self.add_rule(s)\n        if "->" in s:\n            left, right = [x.strip() for x in s.split("->", 1)]\n            head = right\n            body_atoms = [x.strip() for x in re.split(r"[\\*,]", left) if x.strip()]\n            body = ", ".join(body_atoms)\n            return self.add_rule(f"{head} :- {body}.")\n        raise ValueError("axi(...) ожидает \'head :- body\' или \'body -> head\'")\n\n    def alp(self, text: str):\n        # факты (можно несколько как в [a,b] — но для предикатов проще по одному)\n        return self.add_fact(text)\n\n    def list_all(self):\n        if not self.clauses:\n            print("(KB пуста)")\n            return\n        for c in self.clauses:\n            if c.body:\n                print(f"cl#{c.cid}: {pretty_atom(c.head)} :- {\', \'.join(pretty_atom(b) for b in c.body)}.")\n            else:\n                print(f"cl#{c.cid}: {pretty_atom(c.head)}.")\n\n    def solve(self, query_text: str, trace: bool = True, max_solutions: int = 10, max_depth: int = 100):\n        if max_depth < 1 or max_solutions < 1: raise ValueError(\'Лимиты должны быть положительными\')\n        self.max_depth = max_depth\n        self.depth_exceeded = False\n        goals0 = parse_query_atoms(query_text)\n        query_vars = set().union(*(vars_in_atom(g) for g in goals0))\n\n        if trace:\n            print("=== SLD-вывод (трассировка) ===")\n            print("Запрос:", ", ".join(pretty_atom(g) for g in goals0))\n\n        sols = []\n        for sol in self._prove(list(goals0), {}, trace=trace, depth=0):\n            sols.append(sol)\n            if len(sols) >= max_solutions:\n                break\n\n        if self.depth_exceeded:\n            print(\'Достигнут лимит глубины: поиск неполон.\')\n        # печать результата\n        if not sols:\n            print(\'Результат: НЕИЗВЕСТНО (лимит поиска)\' if self.depth_exceeded else \'Результат: ЛОЖЬ (решений нет)\')\n            return []\n\n        # если запрос без переменных (ground), то достаточно одного решения\n        if not query_vars:\n            print("Результат: ИСТИНА")\n            return sols\n\n        print(f"Результат: ИСТИНА (найдено решений: {len(sols)})")\n        for i, s in enumerate(sols, 1):\n            # покажем только переменные исходного запроса\n            items = []\n            for v in sorted(query_vars):\n                tv = apply_subst_term(Var(v), s)\n                items.append(f"{v} = {pretty_term(tv)}")\n            print(f"  Solution #{i}: " + ", ".join(items))\n\n        return sols\n\n    def _prove(self, goals: List[Atom], subst: Subst, trace: bool, depth: int) -> Iterable[Subst]:\n        if not goals:\n            yield subst\n            return\n\n        if depth >= getattr(self, \'max_depth\', 100):\n            self.depth_exceeded = True\n            return\n        # выбираем первую цель\n        goal = apply_subst_atom(goals[0], subst)\n        rest = goals[1:]\n\n        if trace:\n            print("  " * depth + f"- Цель: {pretty_atom(goal)}")\n\n        for cl in self.clauses:\n            # standardize apart\n            cl2 = standardize_apart(cl, self._fresh)\n            self._fresh += 1\n\n            # пробуем унифицировать head с goal\n            s2 = unify_atom(cl2.head, goal, subst)\n            if s2 is None:\n                continue\n\n            # новые цели: body + rest\n            new_goals = [apply_subst_atom(b, s2) for b in cl2.body] + [apply_subst_atom(r, s2) for r in rest]\n\n            if trace:\n                print("  " * depth + f"  Применяем cl#{cl.cid}: {cl.text.strip()}")\n                # покажем изменившиеся подстановки (грубо)\n                show = []\n                for k in sorted(s2.keys()):\n                    show.append(f"{k}={pretty_term(apply_subst_term(Var(k), s2))}")\n                print("  " * depth + f"  Подстановка: {{ " + ", ".join(show) + " }}")\n                if new_goals:\n                    print("  " * depth + "  Новые цели: " + ", ".join(pretty_atom(g) for g in new_goals))\n                else:\n                    print("  " * depth + "  Новые цели: (пусто)")\n\n            yield from self._prove(new_goals, s2, trace=trace, depth=depth+1)\n\ndef pretty_term(t: Term) -> str:\n    if isinstance(t, Var):\n        return t.name\n    if isinstance(t, Const):\n        return t.name\n    if isinstance(t, Func):\n        return f"{t.name}(" + ", ".join(pretty_term(a) for a in t.args) + ")"\n    return str(t)\n\ndef pretty_atom(a: Atom) -> str:\n    if not a.args:\n        return a.pred\n    return f"{a.pred}(" + ", ".join(pretty_term(t) for t in a.args) + ")"\n\n# 3) CLI / команды сценария\n\nHELP_TEXT = """MiniProlog — справка (ПР#2: исчисление предикатов)\n\nКоманды KB:\n  clear\n  fact(p(a,b))\n  rule(h(X) :- b1(X), b2(X)).\n\n  list_kb                  показать все клаузы (факты и правила)\n  goal(q(X)).              выполнить запрос (SLD-вывод), вывести трассировку\n\nБулева алгебра (как в ПР#1):\n  truth(expr)\n  equiv(e1, e2)\n  laws\n  sat(expr)\n  valid(expr)\n\nЗамечания:\n- Переменные: начинаются с заглавной буквы (X, User) или \'_\'\n- Константы: с маленькой (u1, passport, ticket)\n- Предикаты: p(...), can_board(...), has(...)\n"""\n\ndef parse_command(line: str) -> Tuple[str, str]:\n    line = line.strip().removesuffix(\'.\').strip()\n    m = re.match(r"^([A-Za-z_][A-Za-z0-9_]*)\\s*(\\((.*)\\))?$", line)\n    if not m:\n        raise ValueError("Не распознан ввод. Введите help() для справки.")\n    name = m.group(1)\n    has_parens = m.group(2) is not None\n    arg = (m.group(3) or "").strip()\n    if not has_parens:\n        arg = ""\n    return name, arg\n\ndef run_script(path: str):\n    kb = MiniPrologKB()\n\n    with open(path, "r", encoding="utf-8") as f:\n        for raw in f:\n            line = raw.strip()\n            if not line:\n                continue\n            if "#" in line:\n                line = line.split("#", 1)[0].strip()\n                if not line:\n                    continue\n\n            print(f">> {line}")\n            try:\n                name, arg = parse_command(line)\n\n                if name in ("quit", "exit"):\n                    return\n                if name == "help":\n                    print(HELP_TEXT); continue\n                if name == "clear":\n                    kb.clear(); print("Очищено."); continue\n\n                # KB-команды\n                if name == "fact":\n                    kb.add_fact(arg); continue\n                if name == "rule":\n                    kb.add_rule(arg); continue\n                if name == "axi":\n                    kb.axi(arg); continue\n                if name == "alp":\n                    kb.alp(arg); continue\n                if name == "list_kb":\n                    kb.list_all(); continue\n                if name == "goal":\n                    kb.solve(arg, trace=True); continue\n\n                # Булева алгебра\n                if name == "truth":\n                    cmd_truth(arg); continue\n                if name == "equiv":\n                    cmd_equiv(arg); continue\n                if name == "laws":\n                    cmd_laws(); continue\n                if name == "sat":\n                    cmd_sat(arg); continue\n                if name == "valid":\n                    cmd_valid(arg); continue\n\n                raise ValueError(f"Неизвестная функция: {name}")\n\n            except Exception as e:\n                print(f"Ошибка: {e}")\n\ndef main(argv: List[str]) -> int:\n    if len(argv) != 2:\n        print("Использование: python miniprolog.py <script.txt>")\n        return 2\n    run_script(argv[1])\n    return 0\n\nif __name__ == "__main__":\n    raise SystemExit(main(sys.argv))\n', encoding="utf-8")
print('Сохранён miniprolog.py')


## Сценарий MiniProlog


In [ ]:
from pathlib import Path
Path('scenario_pr2_flight.txt').write_text('help\nclear\n\n# --- БЗ в предикатной форме ---\nfact(has(u1, passport))\nfact(has(u1, ticket))\nfact(has(u1, visa))\n\nrule(can_board(U) :- has(U, passport), has(U, ticket).)\nrule(can_fly_international(U) :- can_board(U), has(U, visa).)\n\nlist_kb\n\n# СИТУАЦИЯ 1: все документы есть -> TRUE\ngoal(can_fly_international(u1))\n\n# СИТУАЦИЯ 2: нет визы -> FALSE\nclear\nfact(has(u2, passport))\nfact(has(u2, ticket))\nrule(can_board(U) :- has(U, passport), has(U, ticket).)\nrule(can_fly_international(U) :- can_board(U), has(U, visa).)\ngoal(can_fly_international(u2))\n\n# СИТУАЦИЯ 3: запрос с переменной (подстановка U=...)\nclear\nfact(has(u1, passport))\nfact(has(u1, ticket))\nfact(has(u1, visa))\nfact(has(u3, passport))\nfact(has(u3, ticket))\n# u3 без визы\nrule(can_board(U) :- has(U, passport), has(U, ticket).)\nrule(can_fly_international(U) :- can_board(U), has(U, visa).)\ngoal(can_fly_international(U))\n', encoding="utf-8")
print('Сохранён scenario_pr2_flight.txt')


## Запуск MiniProlog


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, 'miniprolog.py', 'scenario_pr2_flight.txt'], check=True)
